In [136]:
import numpy as np
import pandas as pd

In [137]:
# Load in the dataset
df = pd.read_csv('../../datasets/MBA.csv')

# Must fill nan values based on what dataset calls for
values = {"race": "International", "admission": "Deny"}
df = df.fillna(value=values)

# Drop application id, because its not related to our data
# BTW Drop the international column, because it's already incoded within race column
X = df.drop(columns=['application_id', 'international', 'admission']).to_numpy()
y = df['admission'].to_numpy().reshape(X.shape[0], 1)

# Label which columns are categorical (in X)
categorical_features_idx = [0, 2, 3, 6]

In [138]:
# Split into train and test
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.15, random_state=42, shuffle=True)

In [139]:
# Calculate the gini impurity of a node
def gini(examples: np.ndarray):
    """
    Given an examples array holding the classes of each example,
    calculate the gini impurity of the current node.
    """
    N = len(examples)

    # Get all unique classes
    _, counts = np.unique(examples, return_counts=True)

    # Find proportions and apply gini formula
    probabilities = counts / N

    return 1 - np.sum(probabilities**2)

def weighted_gini(y, mask):
    N = len(y)

    # Get the positive/negative (left/right) examples
    y_left = y[mask]
    y_right = y[~mask]

    # Calculate impurities
    gini_left = gini(y_left)
    gini_right = gini(y_right)

    # Calculate weighted average
    weighted_avg = (len(y_left) / N) * gini_left + (len(y_right) / N) * gini_right

    return weighted_avg

In [140]:
# Node for decision tree
class DTNode:
    def __init__(self, feature=None, threshold=None, mode="numerical", pred=None, left=None, right=None):
        # Store split features
        self.feature = feature
        self.threshold = threshold
        self.mode = mode
        
        self.pred = pred

        # Matches split condition, go left
        self.left: DTNode | None = left

        # Fails split condition, go right
        self.right: DTNode | None = right

    def is_leaf(self):
        return self.left == self.right == None

    # Go down the nodes until find a leaf node--then predict
    def predict(self, x):
        if self.is_leaf():
            return self.pred
        
        if self.mode == "numerical":
            return self.left.predict(x) if x[self.feature] < self.threshold else self.right.predict(x)
        else:
            return self.left.predict(x) if x[self.feature] == self.threshold else self.right.predict(x)

In [141]:
# TODO: Make subsampling of numerical features, because when you have 10,000 midpoints, it gets kinda rough

class DecisionTreeClassifier:
    def __init__(self, max_depth=10, categorical_features=[]):
        self.max_depth = max_depth

        # Initialize the binary tree
        self.root = None

        # Denote certain columns as categorical
        self.categorical_features = categorical_features

    # MODIFIED BEST SPLIT FUNCTION FOR RANDOM FOREST
    def find_best_split(self, X: np.ndarray, y: np.ndarray):
        # Select only a few columns (usually just square root of features)
        N = X.shape[1]
        num_cols_to_avoid = int(np.ceil(np.sqrt(N)))
        columns_to_avoid = np.random.choice(N, num_cols_to_avoid, replace=False)

        # Keep track of best split
        best_gini = np.inf
        best_split_info = None # In the form (feature_index, split_threshold, split_type)

        # Traverse through each feature
        for i, col in enumerate(X.T):
            # Skip the columns that must be avoided
            if i in columns_to_avoid:
                continue

            unique = np.unique(col) # Get all unique elements
            
            # Don't try if there's only one unique element
            if len(unique) == 1:
                continue

            # Categorical type
            is_categorical = (i in self.categorical_features) or isinstance(col[0], str)
            if is_categorical:

                for category in unique:
                    mask = (col == category)

                    # Get weighted avg
                    weighted_avg = weighted_gini(y, mask)

                    # Update best category
                    if weighted_avg < best_gini:
                        best_gini = weighted_avg
                        best_split_info = (i, category, "categorical")
            else:
                # Sort the numerical array unique values
                sorted_vals = np.sort(unique)

                # Calculate pairs of medians:
                medians = (sorted_vals[:-1] + sorted_vals[1:]) / 2

                # Find best split among medians
                for median in medians:
                    mask = (col < median)

                    # Get weighted avg
                    weighted_avg = weighted_gini(y, mask)

                    # Update best category
                    if weighted_avg < best_gini:
                        best_gini = weighted_avg
                        best_split_info = (i, median, "numerical")

        return best_split_info

    def build_tree(self, X: np.ndarray, y: np.ndarray, depth = 1):
        unique, counts = np.unique(y, return_counts=True)
        majority_class = unique[np.argmax(counts)]

        # Don't go past max_depth
        if depth > self.max_depth:
            return DTNode(pred=majority_class)
        
        # If only one category in y, don't go further
        if len(unique) == 1:
            return DTNode(pred=majority_class)
        
        # Find the best split at the current state
        split = self.find_best_split(X, y)

        # If no best split, make leaf
        if split == None:
            return DTNode(pred=majority_class)

        # Create a new node
        feature_idx, threshold, split_type = split
        node = DTNode(feature_idx, threshold, split_type)

        # Find the best splits for left and right node
        if split_type == "categorical":
            mask = (X[:, feature_idx] == threshold)
        else:
            mask = (X[:, feature_idx] < threshold)
        
        # Get examples and features for left and right node
        X_left, y_left = X[mask], y[mask]
        X_right, y_right = X[~mask], y[~mask]

        # Recursively add nodes
        node.left = self.build_tree(X_left, y_left, depth + 1)
        node.right = self.build_tree(X_right, y_right, depth + 1)

        # Return the final node
        return node
    
    def fit(self, X, y):
        """
        Builds optimal decision tree based on data
        """
        self.root = self.build_tree(X, y)

    def predict(self, X):
        """
        Predits a group of examples
        """
        return np.array([self.root.predict(x) for x in X])

In [142]:
# Bootstrapping dataset function
def bootstrap_data(X, y):
    N = X.shape[0]

    # Choose N observations with replacements
    gen = np.random.default_rng() # Don't set seed since we'll be calling this multiple times
    row_indices = gen.integers(low=0, high=N, size=N)

    # Get data from bootstrapped row indices
    bootstrapped_X = X[row_indices]
    bootstrapped_y = y[row_indices]

    return bootstrapped_X, bootstrapped_y

def get_majority_class(y):
    unique, counts = np.unique(y, return_counts=True)
    majority_class = unique[np.argmax(counts)]
    return majority_class

In [143]:
class RandomForestClassifier:
    def __init__(self, max_num_tree, max_depth, categorical_values):
        self.trees = [DecisionTreeClassifier(max_depth=max_depth, categorical_features=categorical_values) for i in range(max_num_tree)]

    def fit(self, X, y):
        for i, tree in enumerate(self.trees):
            # Verbose trainning
            if i % 10 == 0:
                if i != 0:
                    print(". [✔]")
                print(f'Training tree {i+1} to {i+10} ', end="")
            else:
                print(".", end="")

            X_boot, y_boot = bootstrap_data(X, y)
            tree.fit(X_boot, y_boot)

    def predict(self, X):
        # Get predictions from each tree for each observation
        votes = np.array([tree.predict(X) for tree in self.trees])

        # Columns will be the votes for each observation
        predicted_classes = np.apply_along_axis(get_majority_class, axis=0, arr=votes)

        return predicted_classes

In [ ]:
model = RandomForestClassifier(max_num_tree=50, max_depth=7, categorical_values=categorical_features_idx)
model.fit(X_train, y_train)

Training tree 1 to 10 .......... [✔]
Training tree 11 to 20 .......... [✔]
Training tree 21 to 30 .......... [✔]
Training tree 31 to 40 ...

KeyboardInterrupt: 

In [ ]:
# Evaluate the model
y_pred = model.predict(X_test)
flattened_y_test = np.squeeze(y_test, axis=1)

correct = (flattened_y_test == y_pred).astype(int)

accuracy = np.sum(correct) / len(correct)

print(f'Accuracy: {accuracy:.3f}')

Accuracy: 0.826
